In [1]:
import json
import time

import numpy as np
import photolab as lab
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

torch.set_grad_enabled(False)
# "eager" attention is plain matrix maths, which every ONNX runtime understands
model = AutoModel.from_pretrained(lab.MODEL_DIR, attn_implementation="eager").eval()
processor = AutoProcessor.from_pretrained(lab.MODEL_DIR)
OUT = lab.HERE / "onnx"
OUT.mkdir(exist_ok=True)

/Users/rushilshandil/.cache/uv/archive-v0/82JjMR0gVF3nCFwD/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 408/408 [00:00<00:00, 11224.50it/s]


In [2]:
def as_tensor(out):
    return getattr(out, "pooler_output", out)


class ImageTower(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, pixel_values):
        v = as_tensor(self.m.get_image_features(pixel_values=pixel_values))
        return torch.nn.functional.normalize(v, dim=-1)


class TextTower(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids):
        v = as_tensor(self.m.get_text_features(input_ids=input_ids))
        return torch.nn.functional.normalize(v, dim=-1)

In [3]:
size = processor.image_processor.size
example_pixels = torch.zeros(1, 3, size["height"], size["width"])
example_ids = processor(
    text=["a photo of a van"], padding="max_length", max_length=64, return_tensors="pt"
)["input_ids"]
torch.onnx.export(
    ImageTower(model),
    (example_pixels,),
    OUT / "image-fp32.onnx",
    input_names=["pixel_values"],
    output_names=["embedding"],
    dynamic_axes={"pixel_values": {0: "batch"}, "embedding": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)
torch.onnx.export(
    TextTower(model),
    (example_ids,),
    OUT / "text-fp32.onnx",
    input_names=["input_ids"],
    output_names=["embedding"],
    dynamic_axes={"input_ids": {0: "batch"}, "embedding": {0: "batch"}},
    opset_version=17,
    dynamo=False,
)
for f in sorted(OUT.glob("*.onnx")):
    print(f.name, round(f.stat().st_size / 1e6, 1), "MB")

/var/folders/30/6qwmsmzn6w7f11skgbbvbx340000gn/T/ipykernel_92709/582691942.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/var/folders/30/6qwmsmzn6w7f11skgbbvbx340000gn/T/ipykernel_92709/582691942.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/Users/rushilshandil/.cache/uv/arch

image-fp32.onnx 371.8 MB
text-fp32.onnx 441.3 MB


In [4]:
from onnxruntime.quantization import QuantType, quantize_dynamic

for tower in ("image", "text"):
    quantize_dynamic(
        OUT / f"{tower}-fp32.onnx", OUT / f"{tower}-int8.onnx", weight_type=QuantType.QInt8
    )
for f in sorted(OUT.glob("*.onnx")):
    print(f.name, round(f.stat().st_size / 1e6, 1), "MB")

image-fp32.onnx 371.8 MB
image-int8.onnx 99.0 MB
text-fp32.onnx 441.3 MB
text-int8.onnx 111.0 MB


In [5]:
config = processor.image_processor


def preprocess(path):
    image = (
        Image.open(path)
        .convert("RGB")
        .resize((config.size["width"], config.size["height"]), resample=config.resample)
    )
    x = np.asarray(image, dtype=np.float32) * config.rescale_factor
    x = (x - np.array(config.image_mean, dtype=np.float32)) / np.array(
        config.image_std, dtype=np.float32
    )
    return x.transpose(2, 0, 1)  # height, width, colour -> colour, height, width


some_photo = lab.photos_of(sorted({c.client for c in lab.load_cases()})[0])[0]
theirs = processor(images=[Image.open(some_photo).convert("RGB")], return_tensors="np")[
    "pixel_values"
][0]
print("preprocessing differs by at most", float(np.abs(preprocess(some_photo) - theirs).max()))

preprocessing differs by at most 0.007843255996704102


In [6]:
import onnxruntime as ort


def session(name):
    return ort.InferenceSession(str(OUT / name), providers=["CPUExecutionProvider"])


cases = lab.load_cases()
clients = sorted({c.client for c in cases})
paths = [p for c in clients for p in lab.photos_of(c)]
pixels = np.stack([preprocess(p) for p in paths])
ids = processor(
    text=[f"a photo of {c.query}." for c in cases],
    padding="max_length",
    max_length=64,
    return_tensors="np",
)["input_ids"].astype(np.int64)
reference_images = ImageTower(model)(torch.from_numpy(pixels)).numpy()
reference_texts = TextTower(model)(torch.from_numpy(ids)).numpy()
for kind in ("fp32", "int8"):
    got_images = session(f"image-{kind}.onnx").run(None, {"pixel_values": pixels})[0]
    got_texts = session(f"text-{kind}.onnx").run(None, {"input_ids": ids})[0]
    print(
        kind,
        "| worst cosine to PyTorch: photos",
        round(float((got_images * reference_images).sum(1).min()), 5),
        "| queries",
        round(float((got_texts * reference_texts).sum(1).min()), 5),
    )

fp32 | worst cosine to PyTorch: photos 1.0 | queries 1.0
int8 | worst cosine to PyTorch: photos 0.92738 | queries 0.98899


In [7]:
def timed(fn, repeats=10):
    fn()  # warm up
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return (time.perf_counter() - start) / repeats * 1000


for kind in ("fp32", "int8"):
    image_session, text_session = session(f"image-{kind}.onnx"), session(f"text-{kind}.onnx")
    print(
        kind,
        "| one photo",
        round(timed(lambda s=image_session: s.run(None, {"pixel_values": pixels[:1]})), 1),
        "ms",
        "| one query",
        round(timed(lambda s=text_session: s.run(None, {"input_ids": ids[:1]})), 1),
        "ms",
    )

fp32 | one photo 57.9 ms | one query 16.9 ms
int8 | one photo 34.6 ms | one query 10.6 ms


In [8]:
names_by_client = {c: [p.name for p in lab.photos_of(c)] for c in clients}
image_session, text_session = session("image-int8.onnx"), session("text-int8.onnx")
vectors_by_client = {
    c: image_session.run(
        None, {"pixel_values": np.stack([preprocess(p) for p in lab.photos_of(c)])}
    )[0]
    for c in clients
}
scale, bias = model.logit_scale.exp().item(), model.logit_bias.item()


def onnx_search(case):
    ids_one = processor(
        text=[f"a photo of {case.query}."], padding="max_length", max_length=64, return_tensors="np"
    )["input_ids"].astype(np.int64)
    query = text_session.run(None, {"input_ids": ids_one})[0][0]
    cosines = vectors_by_client[case.client] @ query
    return (
        names_by_client[case.client],
        cosines,
        float((1 / (1 + np.exp(-(cosines * scale + bias)))).max()),
    )


print(lab.evaluate(cases, onnx_search))

{'recall@1': 0.9411764705882353, 'recall@3': 1.0, 'mrr': 0.9705882352941176, 'cases': 48.0}


In [9]:
import shutil

parity = {
    "logit_scale": scale,
    "logit_bias": bias,
    "image": {
        "size": [config.size["height"], config.size["width"]],
        "resample": int(config.resample),
        "rescale": config.rescale_factor,
        "mean": list(config.image_mean),
        "std": list(config.image_std),
    },
    "text": {"max_length": 64, "template": "a photo of {}."},
    "queries": [
        {
            "text": f"a photo of {c.query}.",
            "input_ids": row.tolist(),
            "embedding": vec.round(6).tolist(),
        }
        for c, row, vec in zip(cases[:12], ids[:12], reference_texts[:12], strict=True)
    ],
    "photos": [
        {"file": f"{p.parent.name}/{p.name}", "embedding": vec.round(6).tolist()}
        for p, vec in zip(paths[:6], reference_images[:6], strict=True)
    ],
}
(OUT / "parity.json").write_text(json.dumps(parity))
shutil.copy(lab.MODEL_DIR / "spiece.model", OUT / "spiece.model")
print(sorted(f.name for f in OUT.iterdir()))

['image-fp32.onnx', 'image-int8.onnx', 'parity.json', 'spiece.model', 'text-fp32.onnx', 'text-int8.onnx']


In [10]:
from collections import Counter

COST = {"right": 0.0, "needless ask": 1.0, "wrong swap": 3.0}
answers = {(c.client, c.query): onnx_search(c) for c in cases}  # each case searched once


def outcome(case, line):
    names, scores, confidence = answers[(case.client, case.query)]
    if confidence < line:
        return "needless ask" if case.want else "right"
    return "right" if names[int(np.argmax(scores))] in case.want else "wrong swap"


def cheapest_line(group):
    lines = [0.0, *sorted({answers[(c.client, c.query)][2] for c in group}), 1.0]
    return min(lines, key=lambda t: sum(COST[outcome(c, t)] for c in group))


held_out = []
for client in clients:
    line = cheapest_line([c for c in cases if c.client != client])
    held_out += [outcome(c, line) for c in cases if c.client == client]
print("int8, line chosen on the other clients:", dict(Counter(held_out)))
print("line for the product (int8, all cases):", cheapest_line(cases))

int8, line chosen on the other clients: {'right': 35, 'wrong swap': 4, 'needless ask': 9}
line for the product (int8, all cases): 0.03618601709604263


In [11]:
from onnxruntime.quantization import QuantType, quantize_dynamic

for tower in ("image", "text"):
    quantize_dynamic(
        OUT / f"{tower}-fp32.onnx",
        OUT / f"{tower}-int8pc.onnx",
        weight_type=QuantType.QInt8,
        per_channel=True,  # one scale per output column, not one per weight matrix
    )
KINDS = ("fp32", "int8", "int8pc")
image_vectors = {
    k: session(f"image-{k}.onnx").run(None, {"pixel_values": pixels})[0] for k in KINDS
}
text_vectors = {k: session(f"text-{k}.onnx").run(None, {"input_ids": ids})[0] for k in KINDS}
for k in KINDS:
    print(
        k,
        "| worst cosine: photos",
        round(float((image_vectors[k] * reference_images).sum(1).min()), 5),
        "| queries",
        round(float((text_vectors[k] * reference_texts).sum(1).min()), 5),
    )

fp32 | worst cosine: photos 1.0 | queries 1.0
int8 | worst cosine: photos 0.92738 | queries 0.98899
int8pc | worst cosine: photos 0.92033 | queries 0.99492


In [12]:
import pandas as pd

row_of = {p: i for i, p in enumerate(paths)}  # where each photo sits in the arrays above


def searcher(image_kind, text_kind):
    keys = [(c.client, c.query) for c in cases]
    query_of = dict(zip(keys, text_vectors[text_kind], strict=True))

    def search(case):
        rows = [row_of[p] for p in lab.photos_of(case.client)]
        cosines = image_vectors[image_kind][rows] @ query_of[(case.client, case.query)]
        confidence = float((1 / (1 + np.exp(-(cosines * scale + bias)))).max())
        return names_by_client[case.client], cosines, confidence

    return search


grid = []
for image_kind in KINDS:
    for text_kind in KINDS:
        search = searcher(image_kind, text_kind)
        honest = lab.leave_one_client_out(cases, search)
        grid.append(
            {
                "photos": image_kind,
                "queries": text_kind,
                **lab.evaluate(cases, search),
                "honest right action": honest["right_action"],
            }
        )
pd.DataFrame(grid).drop(columns=["cases"]).round(3)

,photos,queries,recall@1,recall@3,mrr,honest right action
0,fp32,fp32,1.000,1.0,1.000,0.833
1,fp32,int8,1.000,1.0,1.000,0.854
2,fp32,int8pc,1.000,1.0,1.000,0.854
3,int8,fp32,0.941,1.0,0.971,0.792
4,int8,int8,0.941,1.0,0.971,0.792
5,int8,int8pc,0.941,1.0,0.971,0.771
6,int8pc,fp32,0.941,1.0,0.971,0.833
7,int8pc,int8,0.941,1.0,0.971,0.771
8,int8pc,int8pc,0.941,1.0,0.971,0.854


In [13]:
chosen = searcher("fp32", "int8pc")  # photos fp32, queries per-channel int8
answers = {(c.client, c.query): chosen(c) for c in cases}  # outcome() and cheapest_line() read this
held_out = []
for client in clients:
    line = cheapest_line([c for c in cases if c.client != client])
    held_out += [outcome(c, line) for c in cases if c.client == client]
print("photos fp32 + queries int8pc, line chosen on the other clients:", dict(Counter(held_out)))
print("line for the product:", cheapest_line(cases))

photos fp32 + queries int8pc, line chosen on the other clients: {'right': 37, 'needless ask': 8, 'wrong swap': 3}
line for the product: 0.020537512376904488
